### 0. Installing & importing libraries

In [ ]:
#if you didn't use any of these libraries, run the following command (remove # ):

# !pip install pandas numpy openpyxl geopandas transliterate geopy

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl  # optional, engine for pd.read_excel
import geopandas as gpd
from transliterate import translit
from geopy.geocoders import Nominatim
import time

In [ ]:
pd.set_option('display.max_columns', None)
np.set_printoptions(threshold=np.inf)

## 1. Traffic accidents data

[Source: Serbian police](https://data.gov.rs/sr/datasets/podatsi-o-saobratshajnim-nezgodama-po-politsijskim-upravama-i-opshtinama/)

pd.read_excel() reads each file from the traffic/ folder and pd.concat() merges them all into a single DataFrame. We then rename columns, split the date_time field into separate date and time columns, convert Serbian-language category values to English, and lowercase text fields for consistency.

##### Getting & merging data

In [ ]:
folder_path = Path("traffic/")
excel_files = list(folder_path.glob("*.xlsx")) # glob() finds all .xlsx files in the folder; pd.concat() stacks them into one DataFrame

accidents_list = []

for file in excel_files:
    df = pd.read_excel(file, header=None) # header=None because the files have no column header row
    accidents_list.append(df)

accidents = pd.concat(accidents_list, ignore_index=True)

In [ ]:
accidents.shape

In [ ]:
accidents.head()

In [ ]:
accidents.columns = ["id", "police_dept", 
                     "municipality", "date_time", 
                     "longitude", "latitude", 
                     "status", "type", "details"] 

accidents.head(1)

##### Cleaning data

In [ ]:
accidents[['date', 'time']] = accidents['date_time'].str.split(',', expand=True) # split "15.03.2023, 14:30" into two columns, then parse each into proper types
accidents['date'] = pd.to_datetime(accidents['date'], format="%d.%m.%Y") # pd.to_datetime() lets us later filter/group by year, month, hour, etc.
accidents['time'] = pd.to_datetime(accidents['time'], format="%H:%M").dt.time

In [ ]:
accidents['year'] = accidents['date'].dt.year

In [ ]:
accidents.dtypes

In [ ]:
accidents['status'].unique() # .unique() shows all raw category values before we replace them

# always do this before .replace() to catch typos or unexpected variants

In [ ]:
accidents['status'] = accidents['status'].replace({
    "Sa povredjenim": "injuries",
    "Sa mat.stetom": "material damage",
    "Sa poginulim": "fatalities"
}, regex=False)

In [ ]:
accidents['type'].unique()

In [ ]:
accidents['type'] = accidents['type'].replace({

    'SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA': 'two vehicles no turning',
    'SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK': 'two vehicles with turning or crossing',
    'SN SA JEDNIM VOZILOM': 'single vehicle',
    'SN SA PARKIRANIM VOZILIMA': 'parked vehicles',
    'SN SA PEŠACIMA': 'with pedestrians',

}, regex=False)

In [ ]:
accidents.head(1)

In [ ]:
accidents['police_dept'] = accidents['police_dept'].str.lower()
accidents['municipality'] = accidents['municipality'].str.lower()

In [ ]:
accidents.head(1)

##### Data analysis

In [ ]:
accidents['year'].value_counts().reset_index().sort_values(by='year', ascending=False)

In [ ]:
accidents['municipality'].value_counts(normalize=True).reset_index().head(10)

In [ ]:
accidents['police_dept'].value_counts(normalize=True).reset_index().head(10)

In [ ]:
accidents['status'].value_counts(normalize=True).reset_index().head(10)

In [ ]:
accidents['type'].value_counts(normalize=True).reset_index().head(10)

In [ ]:
accidents['time'].value_counts(normalize=True).reset_index().head(10)

In [ ]:
ns_accidents = accidents[accidents.police_dept == 'beograd']
ns_accidents.head(2)

In [ ]:
ns_accidents.shape

## 2. School locations data

[Source: Ministry of Education](https://opendata.mpn.gov.rs)

We load the Ministry of Education spreadsheet, transliterate Serbian Cyrillic to Latin script using the transliterate library, drop irrelevant columns, and rename the remaining ones to English. We then filter for Novi Sad schools and geocode their addresses using geopy's Nominatim (OpenStreetMap) to get latitude/longitude coordinates.

##### Cleaning data

In [ ]:
schools = pd.read_excel('elementary schools.xlsx')
schools.head(2)

In [ ]:
schools.iloc[0]

In [ ]:
schools = schools.map(lambda x: translit(x, 'sr', reversed=True) if isinstance(x, str) else x) # translit(..., 'sr', reversed=True) converts Serbian Cyrillic to Latin, lambda applies it cell-by-cell but skips non-string values (numbers, NaN)
schools.head(2)

In [ ]:
schools = schools.drop(columns=[
    "Skolska godina",
    "Skolska uprava",
    "Povrsina objekta",
    "Broj ucionica",
    "Povrsina ucionica",
    "Broj kabinete",
    "Povrsina kabineta",
    "Broj laboratorija",
    "Povrsina laboratorija",
    "Broj radionica",
    "Povrsina radionica",
    "Broj kuhinja",
    "Povrsina kuhinja",
    "Broj restorana u objektu",
    "Broj fiskulturnih sala",
    "Povrsina fiskulturnih sala",
    "Broj biblioteka",
    "Naziv lokacije ",
    "Naziv objekta",
    "Sediste/radna jedinica",
    "Vrsta Ustanove"
])

In [ ]:
schools = schools.rename(columns={
    "Okrug": "district",
    "Mesto/Grad": "city",
    "Opstina": "municipality",
    "Naziv ustanove": "name",
    "Vrsta Osnivaca": "private_public",
    "Adresa": "address"
})

In [ ]:
schools.head(2)

In [ ]:
schools.shape

In [ ]:
schools.dropna(how='all', inplace=True)

In [ ]:
schools.shape

In [ ]:
schools.dtypes

In [ ]:
schools['district'] = schools['district'].str.lower()
schools['city'] = schools['city'].str.lower()
schools['municipality'] = schools['municipality'].str.lower()

In [ ]:
schools.head(2)

In [ ]:
schools['private_public'].unique()

In [ ]:
schools['private_public'] = schools['private_public'].replace({
    "Privatna ustanova": "private",
    "Javna ustanova": "public",
}, regex=False)

In [ ]:
schools.head(2)

In [ ]:
schools['city'].value_counts()

In [ ]:
novi_sad = schools[schools.city == 'novi sad']
novi_sad.shape

##### Geocoding

In [ ]:
novi_sad['address'].unique()

In [ ]:
novi_sad['address'] = novi_sad['address'].str.title()

In [ ]:
geolocator = Nominatim(user_agent="my_app") # Nominatim is a free geocoder powered by OpenStreetMap -- no API key required

latitudes = []
longitudes = []

for idx, row in novi_sad.iterrows():
    location = geolocator.geocode(row['address'])
    if location:
        latitudes.append(location.latitude)
        longitudes.append(location.longitude)
    else:
        latitudes.append(None) # if location is None the address wasn't found, we store NaN and check later
        longitudes.append(None)
    time.sleep(1)  # time.sleep(1) is mandatory: Nominatim's usage policy requires max 1 request/second

novi_sad = novi_sad.copy()
novi_sad["latitude"] = latitudes
novi_sad["longitude"] = longitudes

novi_sad.head(2)

In [ ]:
novi_sad['name'] = novi_sad['name'].str.replace('"', '', regex=False)
novi_sad.head(2)

In [ ]:
novi_sad[novi_sad['latitude'].isna()]

In [ ]:
novi_sad[novi_sad['longitude'].isna()]

In [ ]:
print(novi_sad["longitude"].min(), novi_sad["longitude"].max())
print(novi_sad["latitude"].min(), novi_sad["latitude"].max())

### 3. Geo DFs

Both cleaned DataFrames are converted into GeoDataFrames using geopandas, with geometry built from their coordinate columns. Both are exported as GeoJSON files, ready to be loaded into mapping tools like QGIS, Datawrapper, or Flourish.

In [ ]:
ns_accidents_gdf = gpd.GeoDataFrame(
    ns_accidents,
    geometry=gpd.points_from_xy(ns_accidents['longitude'], ns_accidents['latitude']),
    crs="EPSG:4326" # EPSG:4326 is standard WGS84, the same coordinate system used by GPS and most web maps
) 

# GeoDataFrame wraps a regular DataFrame and adds a geometry column

In [ ]:
novi_sad_gdf = gpd.GeoDataFrame(
    novi_sad,
    geometry=gpd.points_from_xy(novi_sad['longitude'], novi_sad['latitude']),
    crs="EPSG:4326"
)

In [ ]:
ns_accidents_gdf.to_file("ns_accidents.geojson", driver="GeoJSON")
novi_sad_gdf.to_file("ns_schools.geojson", driver="GeoJSON") # GeoJSON is the standard format for geographic data in journalism tools (Datawrapper, Flourish, QGIS)